# ViFinQA — Kaggle GPU: pinned Qwen3-AWQ grounded generation

1. Chọn GPU (T4 x2 dùng được) và bật Internet; attach `vifinqa`, `vifinqa-artifacts`, cùng dense artifact đã tạo bởi notebook build.
2. Nếu GitHub repo còn private, tạo Kaggle Secret tên `GITHUB_TOKEN` có quyền read-only Contents và bật quyền dùng secret cho notebook.
3. Chạy thủ công tới D1/D2 rồi smoke. Ở vòng kế tiếp, bỏ qua hai cell wide/sample và cell full; bật riêng `VIFINQA_RUN_SUBSET_200=1` để chạy matched ablation. Chỉ cân nhắc full sau khi đã đọc các kết quả này.

Kaggle tự giải nén ZIP dataset. Smoke dùng 5 ID cố định đại diện cho nhiều đơn vị và độ phức tạp; full run resume cùng checkpoint.

In [ ]:
# Pinned profiles verified before the competition cutoff. Edit PROFILE, EXPECTED SHA and run flags only.
import os

MODEL_PROFILES = {
    "qwen3_8b_awq": {
        "model": "Qwen/Qwen3-8B-AWQ",
        "revision": "4da05a8edb55c6046cce958586c33b61da07bb79",
        "total_params_b": "8.2",
        "non_embedding_params_b": "6.95",
        "max_num_seqs": "4",
    },
    "qwen3_14b_awq": {
        "model": "Qwen/Qwen3-14B-AWQ",
        "revision": "31c69efc29464b6bb0aee1398b5a7b50a99340c3",
        # Official card: 14.8B total, 13.2B non-embedding. FINAL_RUN is blocked
        # unless the organiser explicitly confirms that this satisfies <=14B.
        "total_params_b": "14.8",
        "non_embedding_params_b": "13.2",
        "max_num_seqs": "2",
    },
}
# Organiser eligibility for the 14B-labelled model was confirmed on 13/08/2026, but it does not
# fit this hardware. On a T4 at --gpu-memory-utilization 0.90 the 14B weights take about 8.6 of
# the ~10.8 GiB budget, leaving 2.18 GiB of KV cache against the 2.50 GiB one 16,384-token
# sequence needs, so the engine refuses to start at all (14/08, RUN-023). The 8B weights take
# 4.8 GiB and leave 6.0 GiB, which holds about 2.7 full-length sequences.
os.environ["VIFINQA_MODEL_PROFILE"] = "qwen3_8b_awq"
profile = MODEL_PROFILES[os.environ["VIFINQA_MODEL_PROFILE"]]
os.environ["VIFINQA_MODEL"] = profile["model"]
os.environ["VIFINQA_MODEL_REVISION"] = profile["revision"]
os.environ["VIFINQA_MODEL_TOTAL_PARAMS_B"] = profile["total_params_b"]
os.environ["VIFINQA_MODEL_NON_EMBEDDING_PARAMS_B"] = profile["non_embedding_params_b"]
os.environ["VIFINQA_MAX_NUM_SEQS"] = profile["max_num_seqs"]
# The 40-character commit printed locally after pushing.
#
# RUN_MODE "smoke" may leave this empty: it runs whatever VIFINQA_GIT_REF points at and records
# the resolved SHA either way. **"subset" and "full" require it**, and the cells below refuse to
# start without it -- which is a guard worth having on a submission candidate, but it stopped a
# session mid-flight once because this comment said empty was fine and those cells disagreed.
#
# The awkwardness is real: committing the notebook moves the SHA, so the value here can never be
# this notebook's own commit. Workflow that works: push, read the SHA, paste it here, re-run this
# cell only. Do not commit the pasted value.
os.environ["VIFINQA_EXPECTED_PROJECT_SHA"] = ""
# Push to this branch/ref; Kaggle checks out its tip and then verifies the SHA above.
os.environ["VIFINQA_GIT_REF"] = "main"
# Keep the written organiser confirmation with the competition working notes.
os.environ["VIFINQA_ORGANIZER_CONFIRMED_14B"] = "1"
os.environ["VIFINQA_DENSE_REVISION"] = "5617a9f61b028005a4858fdac845db406aefb181"
os.environ["VIFINQA_RERANKER_REVISION"] = "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"
os.environ["VIFINQA_THINKING_MODE"] = "disabled"
os.environ["VIFINQA_TABLE_UNIT_SOURCE"] = "latest"
# One switch decides what this session does. The three flags below used to be set by hand and
# had to agree with each other and with QUESTION_LIMIT; a full run that silently stopped at 600
# questions was one forgotten edit away.
RUN_MODE = "full"  # smoke | subset | full
# Running every cell top to bottom already passes through the decoding benchmark and the
# five-question smoke gate before the full run starts, so "full" means the whole sequence
# rather than skipping the checks.
_RUN_MODES = {
    # smoke: five known questions plus the decoding benchmark. Always run this first on a new
    # model or a new machine -- it is what tells you whether the throughput is sane.
    "smoke": {"final": "0", "full": "0", "subset": "0"},
    # subset: the fixed 200-question comparison set, for measuring one change at a time.
    "subset": {"final": "1", "full": "0", "subset": "1"},
    # full: the whole 1,012, split across sessions by QUESTION_LIMIT below.
    "full": {"final": "1", "full": "1", "subset": "0"},
}
_mode = _RUN_MODES[RUN_MODE]
os.environ["VIFINQA_FINAL_RUN"] = _mode["final"]
os.environ["VIFINQA_RUN_FULL"] = _mode["full"]
os.environ["VIFINQA_RUN_SUBSET_200"] = _mode["subset"]
# Run one unit branch per saved Kaggle version. Use manifest first, then latest.
os.environ["VIFINQA_UNIT_VARIANTS"] = "manifest"
# Safety gate: do not start a branch projected above 8h unless you deliberately override.
os.environ["VIFINQA_ALLOW_LONG_SUBSET"] = "0"
# The diagnostics answered their questions and are not free to repeat. The widest-route gate
# alone cost 2.5 hours of a 12-hour session on 23 questions, and the decoding benchmark settled
# the AWQ-kernel question back on 14/08. Set to "1" only on new hardware or a new model, where
# the throughput is unknown again. The smoke always runs: it is 0.35h and it is the regression
# test that catches a broken pipeline before the run.
os.environ["VIFINQA_RUN_DIAGNOSTICS"] = "0"
# Questions 399 and 442 stopped at exactly 2,048 completion tokens on every attempt and
# failed to parse mid-JSON, so the budget was the ceiling rather than the model. At the
# Marlin kernel's 22 tokens per second 4,096 costs 186 s inside a 360 s timeout, and the
# longest prompt measured is 9,337 tokens, so prompt plus budget stays under 16,384.
os.environ["VIFINQA_MAX_TOKENS"] = "6144"
os.environ["VIFINQA_TP"] = "1"
os.environ["VIFINQA_DP"] = "2"
os.environ["VIFINQA_SHARDS_PER_REPLICA"] = "4"
# Answer only the first N questions and stop cleanly. A cancelled session cannot be
# attached as an input to the notebook that finishes the run, so a session that will not
# reach 1,012 in time is worth ending on purpose. Empty means answer all of them.
# Kaggle keeps nothing from a session it kills, so a run that will not reach 1,012 inside the
# twelve-hour cap has to stop on purpose instead. Notebook 02 answers this many and saves; the
# saved version becomes the input to notebook 03, which finishes the rest with no cap of its own.
# Raise it only after a session proves it has the headroom.
os.environ["VIFINQA_QUESTION_LIMIT"] = "600"
print(
    "mode/profile/model/final/full/subset200/tp/dp/git-ref/expected-sha:",
    RUN_MODE,
    os.environ["VIFINQA_MODEL_PROFILE"],
    os.environ["VIFINQA_MODEL"],
    os.environ["VIFINQA_FINAL_RUN"],
    os.environ["VIFINQA_RUN_FULL"],
    os.environ["VIFINQA_RUN_SUBSET_200"],
    os.environ["VIFINQA_TP"],
    os.environ["VIFINQA_DP"],
    os.environ["VIFINQA_GIT_REF"],
    os.environ["VIFINQA_EXPECTED_PROJECT_SHA"] or "<diagnostic-only>",
)

In [ ]:
import base64
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import requests
import torch

def iter_input_paths(
    relative: str, root: Path = Path("/kaggle/input"), max_depth: int = 12
) -> list[Path]:
    """Return every existing `<directory>/relative` under `root`, following symlinked mounts."""
    matches: list[Path] = []
    visited: set[str] = set()
    for parent, directories, _ in os.walk(root, followlinks=True):
        real = os.path.realpath(parent)
        if real in visited:
            directories.clear()
            continue
        visited.add(real)
        if len(Path(parent).parts) - len(root.parts) >= max_depth:
            directories.clear()
        candidate = Path(parent) / relative
        if candidate.exists():
            matches.append(candidate)
    return sorted(matches, key=str)


def describe_inputs(root: Path = Path("/kaggle/input"), max_depth: int = 5) -> str:
    """Return a compact inventory of mounted inputs so failures name what is actually attached.

    Kaggle spends three levels on `datasets/<owner>/<slug>` before any content, so the
    default depth has to reach past the mount point itself.
    """
    if not root.is_dir():
        return f"{root} does not exist"
    lines: list[str] = []
    visited: set[str] = set()
    for parent, directories, filenames in os.walk(root, followlinks=True):
        real = os.path.realpath(parent)
        if real in visited:
            directories.clear()
            continue
        visited.add(real)
        depth = len(Path(parent).parts) - len(root.parts)
        entries = sorted(filenames)[:4]
        if len(filenames) > 4:
            entries.append(f"+{len(filenames) - 4} more files")
        if depth >= max_depth:
            if directories:
                entries.append(f"+{len(directories)} more directories")
            directories.clear()
        directories.sort()
        lines.append(f"{'  ' * depth}{Path(parent).name or root}/ {entries}")
        if len(lines) >= 80:
            lines.append("... truncated")
            break
    return "\n".join(lines)

try:
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    del hf_token
    print("authenticated Hugging Face downloads enabled")

print(
    "torch", torch.__version__, "cuda", torch.cuda.is_available(), "gpus", torch.cuda.device_count()
)
assert torch.cuda.is_available(), "Enable a GPU accelerator before continuing."
requested_dp = int(os.environ.get("VIFINQA_DP", "2"))
assert (
    torch.cuda.device_count() >= requested_dp
), f"VIFINQA_DP={requested_dp} requires at least {requested_dp} GPUs; select T4 x2."
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    capability = torch.cuda.get_device_capability(i)
    print(i, p.name, round(p.total_memory / 2**30, 1), "GiB", "sm", capability)
    assert capability >= (
        7,
        5,
    ), f"GPU {i} {p.name} has capability {capability}; select T4 x2 (sm_75 or newer)."
FINAL_RUN = os.environ.get("VIFINQA_FINAL_RUN") == "1"
RUN_FULL = os.environ.get("VIFINQA_RUN_FULL") == "1"
RUN_SUBSET_200 = os.environ.get("VIFINQA_RUN_SUBSET_200") == "1"
assert sum((RUN_FULL, RUN_SUBSET_200)) <= 1, "Full and subset modes are mutually exclusive."
assert not FINAL_RUN or RUN_FULL, "FINAL_RUN=1 is only valid together with RUN_FULL=1."
MODEL_PROFILE = os.environ["VIFINQA_MODEL_PROFILE"]
MODEL_RUN_TAG = MODEL_PROFILE.replace("-", "_")
MODEL = os.environ["VIFINQA_MODEL"]
MODEL_REVISION = os.environ.get("VIFINQA_MODEL_REVISION")
MODEL_TOTAL_PARAMS_B = float(os.environ["VIFINQA_MODEL_TOTAL_PARAMS_B"])
MODEL_NON_EMBEDDING_PARAMS_B = float(
    os.environ["VIFINQA_MODEL_NON_EMBEDDING_PARAMS_B"]
)
DENSE_REVISION = os.environ.get("VIFINQA_DENSE_REVISION")
RERANKER = "BAAI/bge-reranker-v2-m3"
RERANKER_REVISION = os.environ.get("VIFINQA_RERANKER_REVISION")
THINKING_MODE = os.environ.get("VIFINQA_THINKING_MODE", "disabled")
TABLE_UNIT_SOURCE = os.environ.get("VIFINQA_TABLE_UNIT_SOURCE", "latest")
MAX_TOKENS = os.environ.get("VIFINQA_MAX_TOKENS", "4096")
assert MAX_TOKENS.isdigit() and int(MAX_TOKENS) > 0, "VIFINQA_MAX_TOKENS must be a positive integer."
EXPECTED_PROJECT_SHA = os.environ.get("VIFINQA_EXPECTED_PROJECT_SHA", "").strip()
ORGANIZER_CONFIRMED_14B = os.environ.get("VIFINQA_ORGANIZER_CONFIRMED_14B") == "1"
assert THINKING_MODE in {"disabled", "auto"}
assert TABLE_UNIT_SOURCE in {"manifest", "latest"}
if EXPECTED_PROJECT_SHA:
    assert len(EXPECTED_PROJECT_SHA) == 40 and all(
        character in "0123456789abcdef" for character in EXPECTED_PROJECT_SHA
    ), "VIFINQA_EXPECTED_PROJECT_SHA must be a full lowercase Git SHA."
if FINAL_RUN:
    if MODEL_TOTAL_PARAMS_B > 14:
        assert ORGANIZER_CONFIRMED_14B, (
            "Qwen3-14B reports 14.8B total parameters. Use the 8B profile or set "
            "VIFINQA_ORGANIZER_CONFIRMED_14B=1 only with written organiser approval."
        )
    for name, revision in {
        "model": MODEL_REVISION,
        "dense": DENSE_REVISION,
        "reranker": RERANKER_REVISION,
    }.items():
        valid_revision = (
            revision and len(revision) == 40 and all(c in "0123456789abcdef" for c in revision)
        )
        assert valid_revision, f"Final run requires a full lowercase commit SHA for {name}."

In [ ]:
# Prefer attached code. Otherwise clone anonymously, then use a Kaggle GITHUB_TOKEN secret.
GIT_URL = "https://github.com/ThanhDatVN/AI-Financial-Data-Assistant.git"
GIT_REF = os.environ.get("VIFINQA_GIT_REF", "main").strip()
assert GIT_REF and not GIT_REF.startswith("-"), "Invalid VIFINQA_GIT_REF."
PROJECT = Path("/kaggle/working/AI-Financial-Data-Assistant")
assert PROJECT.parent == Path("/kaggle/working")


def remove_partial_checkout() -> None:
    if PROJECT.exists() and not (PROJECT / "pyproject.toml").exists():
        shutil.rmtree(PROJECT)


def clone_repo() -> None:
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_URL, str(PROJECT)],
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        return
    remove_partial_checkout()
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None
    if not token:
        detail = (result.stderr or "git clone failed").strip().splitlines()[-1]
        raise RuntimeError(
            f"{detail} Enable Internet and either make the repo public, attach the code as a "
            "Kaggle Dataset, or add a read-only GITHUB_TOKEN under Add-ons > Secrets."
        )
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env = os.environ.copy()
    git_env.update(
        {
            "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.extraHeader",
            "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
        }
    )
    authenticated = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_URL, str(PROJECT)],
        env=git_env,
        capture_output=True,
        text=True,
    )
    del token, auth, git_env
    if authenticated.returncode != 0:
        remove_partial_checkout()
        raise RuntimeError("Authenticated clone failed; verify the read-only GITHUB_TOKEN.")


remove_partial_checkout()
attached_candidates = sorted(
    {
        marker.parent
        for marker in iter_input_paths("pyproject.toml")
        if (marker.parent / "scripts/50_generate_programs.py").is_file()
    },
    key=str,
)
if not PROJECT.exists():
    if attached_candidates:
        shutil.copytree(attached_candidates[0], PROJECT)
    else:
        clone_repo()
if (PROJECT / ".git").exists():
    subprocess.run(
        ["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", GIT_REF],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT), "checkout", "--detach", "FETCH_HEAD"],
        check=True,
    )
assert (PROJECT / "pyproject.toml").exists(), f"Invalid project checkout: {PROJECT}"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-gpu.txt")],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)], check=True)
subprocess.run(
    [
        sys.executable,
        "-c",
        "import bm25s, faiss, ftfy, lxml, openai, pandas, pyarrow, rapidfuzz; "
        "import sentence_transformers, Stemmer, unidecode",
    ],
    check=True,
)
package_probe = (
    "import importlib.metadata as m; import torch; "
    "print('resolved runtime:', 'torch', torch.__version__, 'vllm', m.version('vllm'), "
    "'sentence-transformers', m.version('sentence-transformers'), "
    "'transformers', m.version('transformers'), 'openai', m.version('openai')); "
    "assert torch.__version__.startswith('2.10.'); "
    "assert m.version('vllm') == '0.19.1'; "
    "assert m.version('sentence-transformers') == '5.5.1'; "
    "assert m.version('transformers') == '5.5.3'; "
    "assert m.version('openai').split('.')[0] == '2'"
)
subprocess.run([sys.executable, "-c", package_probe], check=True)
os.chdir(PROJECT)
PROJECT_SHA = (
    subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    if (PROJECT / ".git").exists()
    else "attached-archive-no-git-sha"
)
if EXPECTED_PROJECT_SHA:
    assert PROJECT_SHA == EXPECTED_PROJECT_SHA, (
        f"Kaggle checked out {PROJECT_SHA}, expected {EXPECTED_PROJECT_SHA}. "
        "Stop: this is not the published experiment snapshot."
    )
if RUN_SUBSET_200 or FINAL_RUN:
    assert len(PROJECT_SHA) == 40, "Subset/final runs require a Git checkout with a SHA."
    if not EXPECTED_PROJECT_SHA:
        print(
            f"no SHA was declared; this run is pinned to {PROJECT_SHA} from ref {GIT_REF}. "
            "Record it with the results."
        )
print("project revision:", PROJECT_SHA)
RUNTIME_LOG = Path("/kaggle/working/runtime_environment.txt")
RUNTIME_LOG.write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True),
    encoding="utf-8",
)

In [ ]:
# Discover inputs by their contents, because Kaggle dataset slugs and nesting can vary.
INPUT_INVENTORY = describe_inputs()
print("Kaggle inputs:\n" + INPUT_INVENTORY)
data_candidates = sorted(
    {
        questions.parent.parent
        for questions in iter_input_paths("questions/questions.jsonl")
        if (questions.parent.parent / "code_stock.csv").is_file()
        and (questions.parent.parent / "financial_statements").is_dir()
    },
    key=str,
)
assert data_candidates, (
    "ViFinQA files are not mounted in this Kaggle session. "
    "Attach the `vifinqa` input and restart the session if needed.\n"
    f"{INPUT_INVENTORY}"
)
DATA_ROOT = data_candidates[0]
manifest_candidates = sorted(
    {
        manifest
        for manifest in iter_input_paths("processed/table_manifest.jsonl")
        if manifest.with_suffix(".parquet").is_file()
    },
    key=str,
)
assert manifest_candidates, (
    "Frozen retrieval artifacts are not mounted. "
    "Attach the `vifinqa-artifacts` input.\n"
    f"{INPUT_INVENTORY}"
)
MANIFEST = manifest_candidates[0]
ARTIFACT_INPUT = MANIFEST.parents[2]
BM25 = ARTIFACT_INPUT / "data/index/bm25"
assert MANIFEST.with_suffix(".parquet").exists(), "The generation stage needs the Parquet manifest."
assert BM25.exists(), "Attach or build the BM25 index."


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


expected_hashes = {
    MANIFEST: "ced1d671d6a71c299fea02d7d12b86b596b430a2714ab9aeaa4b338fe012fac1",
    MANIFEST.with_suffix(".parquet"): (
        "060bd26eff14d30ce70b3ba7b00af509be6100b58ddb5a0fe970afa0ef69e29d"
    ),
    MANIFEST.with_suffix(".metadata.json"): (
        "bb338c8a01381241a915517fe774b045cc53213eddd4194f645473616c188e12"
    ),
    BM25 / "data.csc.index.npy": "3f2d7292960e8fda6ca5a1f09d10f692259629ac66cc61703b275b927d5cd683",
    BM25 / "indices.csc.index.npy": (
        "6f5ac7fa7be96f946eced6dcfa7eddeb63493d317f13ef4249bc287e3d993991"
    ),
    BM25 / "indptr.csc.index.npy": (
        "6c136f30e633641b65b26ba2341a0a0aedbba2932aa1d51b4b48e953fbf247eb"
    ),
    BM25 / "params.index.json": "b42a70b508494aa5bb40ef81323a35b4b4ed5afbb7984ed224bb700197a01e7c",
    BM25 / "records.jsonl": "1b8ebe896b92e77c71e5ebadb2e519b377932e1b5a9665080e457fce12daf40f",
    BM25 / "vocab.index.json": "3c5482d9e193cb4240e329de6406779ee5a67f1b4f570bde0d4c397aa297f5aa",
}
for path, expected in expected_hashes.items():
    assert path.exists(), f"Missing frozen artefact: {path}"
    actual = sha256(path)
    assert actual == expected, f"SHA-256 mismatch for {path}: {actual}"
question_count = sum(
    1 for line in (DATA_ROOT / "questions/questions.jsonl").open(encoding="utf-8") if line.strip()
)
assert question_count == 1012, f"Expected 1,012 questions, found {question_count}"
print("inputs verified:", DATA_ROOT, MANIFEST, BM25, "questions=", question_count)

In [ ]:
# Reuse the immutable dense Dataset produced by 01_kaggle_build_dense_artifact.ipynb.
dense_artifacts = []
for artifact_path in iter_input_paths("artifact_manifest.json"):
    try:
        metadata = json.loads(artifact_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        continue
    if (
        metadata.get("artifact_type") == "vifinqa_bge_m3_dense_index"
        and metadata.get("model_revision") == DENSE_REVISION
        and metadata.get("tables") == 146246
    ):
        dense_artifacts.append((artifact_path.parent, metadata))
assert dense_artifacts, (
    "Attach the `vifinqa-dense-bge-m3` Dataset produced by the build notebook; "
    "generation intentionally refuses to rebuild the 146,246-table index.\n"
    f"{INPUT_INVENTORY}"
)
DENSE_ARTIFACT_ROOT, dense_artifact = sorted(dense_artifacts, key=lambda item: str(item[0]))[0]
DENSE = DENSE_ARTIFACT_ROOT / "data/index/bge_m3"
assert dense_artifact.get("source_manifest_sha256") == expected_hashes[MANIFEST]
for relative, file_metadata in dense_artifact.get("files", {}).items():
    artifact_file = DENSE_ARTIFACT_ROOT / relative
    assert artifact_file.exists(), f"Missing dense artifact file: {artifact_file}"
    assert sha256(artifact_file) == file_metadata["sha256"], artifact_file
dense_config = json.loads((DENSE / "config.json").read_text(encoding="utf-8"))
assert (
    dense_config.get("tables") == 146246
), f"Dense index must contain 146,246 tables: {dense_config}"
assert dense_config.get("max_seq_length") == 8192, dense_config
assert dense_config.get("use_fp16") is False, dense_config
assert dense_config.get("model_revision") == DENSE_REVISION, dense_config
gpu_free_bytes = [torch.cuda.mem_get_info(index)[0] for index in range(torch.cuda.device_count())]
DENSE_GPU = max(range(len(gpu_free_bytes)), key=gpu_free_bytes.__getitem__)
assert gpu_free_bytes[DENSE_GPU] / 2**30 >= 3, "Dense query encoding needs 3 GiB free."
DENSE_QUERY_DEVICE = f"cuda:{DENSE_GPU}"
print("reusing verified dense artifact:", DENSE, "query device:", DENSE_QUERY_DEVICE)
HYBRID = Path("/kaggle/working/artifacts/retrieval_hybrid.jsonl")
HYBRID_CHECKPOINT = HYBRID.with_name(HYBRID.name + ".checkpoint")


def checkpoint_project_revision(metadata_path: Path) -> str:
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    revision = metadata.get("project_revision")
    valid = (
        isinstance(revision, str)
        and len(revision) == 40
        and all(character in "0123456789abcdef" for character in revision)
    )
    assert valid, f"Invalid checkpoint project revision: {metadata_path}"
    return revision


prior_hybrid = iter_input_paths("retrieval_hybrid.jsonl.checkpoint/run_metadata.json")
if not HYBRID_CHECKPOINT.exists() and prior_hybrid:
    shutil.copytree(prior_hybrid[0].parent, HYBRID_CHECKPOINT)
    print("imported prior hybrid checkpoints:", prior_hybrid[0].parent)
HYBRID_PROJECT_SHA = PROJECT_SHA
hybrid_metadata_path = HYBRID_CHECKPOINT / "run_metadata.json"
if hybrid_metadata_path.exists():
    HYBRID_PROJECT_SHA = checkpoint_project_revision(hybrid_metadata_path)
    print("resuming hybrid artifact revision:", HYBRID_PROJECT_SHA)
subprocess.run(
    [
        sys.executable,
        "scripts/30_retrieve_questions.py",
        "--questions",
        str(DATA_ROOT / "questions/questions.jsonl"),
        "--companies",
        str(DATA_ROOT / "code_stock.csv"),
        "--bm25",
        str(BM25),
        "--dense",
        str(DENSE),
        "--dense-device",
        DENSE_QUERY_DEVICE,
        "--output",
        str(HYBRID),
        "--project-revision",
        HYBRID_PROJECT_SHA,
        "--candidate-k",
        "2000",
        "--top-k",
        "100",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "scripts/32_validate_retrieval.py",
        "--questions",
        str(DATA_ROOT / "questions/questions.jsonl"),
        "--manifest",
        str(MANIFEST.with_suffix(".parquet")),
        "--retrieval",
        str(HYBRID),
        "--output",
        "/kaggle/working/artifacts/retrieval_hybrid_qc.json",
    ],
    check=True,
)
hybrid_qc = json.loads(
    Path("/kaggle/working/artifacts/retrieval_hybrid_qc.json").read_text(encoding="utf-8")
)
assert hybrid_qc.get("passed") is True and hybrid_qc.get("rows") == 1012, hybrid_qc
# Reranking costs hours of cross-encoding and has never been measured against labels, so the
# hybrid ranking stands on its own until something shows the reranker earns that time.
RETRIEVAL = HYBRID
print("hybrid retrieval verified:", RETRIEVAL, hybrid_qc["retrieval_sha256"])
print("Measure on this first; run the reranker cell only to spend hours on the ordering.")


In [ ]:
# Optional: cross-encode on both T4s as disjoint resume-safe shards, then release both
# GPUs for vLLM. Hours of compute whose value against labels is still unmeasured.
from huggingface_hub import snapshot_download  # noqa: E402

snapshot_download(RERANKER, revision=RERANKER_REVISION)
RERANK_SHARDS = Path("/kaggle/working/artifacts/retrieval_rerank_shards")
prior_rerank = iter_input_paths("retrieval_rerank_shards/shard_0/run_metadata.json")
if not RERANK_SHARDS.exists() and prior_rerank:
    prior_root = prior_rerank[0].parents[1]
    shutil.copytree(prior_root, RERANK_SHARDS)
    print("imported prior reranker checkpoints:", prior_root)
RERANK_PROJECT_SHA = PROJECT_SHA
rerank_metadata_path = RERANK_SHARDS / "shard_0/run_metadata.json"
if rerank_metadata_path.exists():
    RERANK_PROJECT_SHA = checkpoint_project_revision(rerank_metadata_path)
    print("resuming reranker artifact revision:", RERANK_PROJECT_SHA)
RERANK_DP = min(2, torch.cuda.device_count())
assert RERANK_DP == 2, "Full reranking is configured for T4 x2."
rerank_workers = []
rerank_shard_dirs = [RERANK_SHARDS / f"shard_{index}" for index in range(RERANK_DP)]
for shard_index, shard_dir in enumerate(rerank_shard_dirs):
    rerank_cmd = [
        sys.executable,
        "scripts/33_rerank_retrieval.py",
        "--retrieval",
        str(HYBRID),
        "--records",
        str(DENSE / "records.jsonl"),
        "--output",
        str(shard_dir),
        "--model",
        RERANKER,
        "--model-revision",
        RERANKER_REVISION,
        "--project-revision",
        RERANK_PROJECT_SHA,
        "--device",
        f"cuda:{shard_index}",
        "--fp16",
        "--max-length",
        "8192",
        "--batch-size",
        "4",
        "--max-batch-tokens",
        "8192",
        "--candidate-tables",
        "100",
        "--top-k",
        "40",
        "--shard-count",
        str(RERANK_DP),
        "--shard-index",
        str(shard_index),
    ]
    if FINAL_RUN:
        rerank_cmd += ["--final-run"]
    rerank_workers.append((shard_index, subprocess.Popen(rerank_cmd)))
for shard_index, worker in rerank_workers:
    return_code = worker.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, ["reranker-shard", str(shard_index)])
RETRIEVAL = Path("/kaggle/working/artifacts/retrieval_reranked.jsonl")
subprocess.run(
    [
        sys.executable,
        "scripts/34_merge_retrieval_shards.py",
        *[str(path) for path in rerank_shard_dirs],
        "--output",
        str(RETRIEVAL),
        "--expected-rows",
        "1012",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "scripts/32_validate_retrieval.py",
        "--questions",
        str(DATA_ROOT / "questions/questions.jsonl"),
        "--manifest",
        str(MANIFEST.with_suffix(".parquet")),
        "--retrieval",
        str(RETRIEVAL),
        "--output",
        "/kaggle/working/artifacts/retrieval_reranked_qc.json",
    ],
    check=True,
)
retrieval_qc_path = Path("/kaggle/working/artifacts/retrieval_reranked_qc.json")
retrieval_qc = json.loads(retrieval_qc_path.read_text(encoding="utf-8"))
assert retrieval_qc.get("passed") is True and retrieval_qc.get("rows") == 1012, retrieval_qc
print("reranked retrieval verified:", retrieval_qc_path, retrieval_qc["retrieval_sha256"])

In [ ]:
# Replicate the model once per T4; full generation sends two independent shards.
#
# Qwen3-14B-AWQ is the tighter fit. Weights land near 8.6 GiB of the 13.8 GiB that
# --gpu-memory-utilization 0.90 leaves on a T4, and its KV cache costs 160 KiB per token
# (40 layers x 8 KV heads x 128 head_dim x 2 x fp16). That leaves room for roughly 34k
# cached tokens, so --max-model-len 16384 fits about two sequences at once rather than the
# four Qwen3-8B-AWQ afforded. MAX_NUM_SEQS defaults down accordingly; raise it only after a
# smoke run shows headroom, because over-subscribing makes vLLM preempt and recompute.
TP = int(os.environ.get("VIFINQA_TP", "1"))
MAX_NUM_SEQS = int(os.environ.get("VIFINQA_MAX_NUM_SEQS", "2"))
DP = int(os.environ.get("VIFINQA_DP", "2"))
assert TP in {1, 2}, "Tensor parallelism above 2 has no second pair of T4s to use."
assert 1 <= DP <= torch.cuda.device_count(), "VIFINQA_DP must not exceed the GPU count."
# Two client shards per replica keep a batch forming while one shard validates and
# executes the program it just received.
SHARDS = DP * int(os.environ.get("VIFINQA_SHARDS_PER_REPLICA", "2"))
# The generator sizes each question's token budget against this, so the server and
# the client must read the same number rather than two copies of it.
MAX_MODEL_LEN = 16384
VLLM_BASE = "http://127.0.0.1:8000"
VLLM_CONFIG = Path("/kaggle/working/vllm_server_config.json")
expected_server_config = {
    "model": MODEL,
    "model_revision": MODEL_REVISION,
    "tensor_parallel_size": TP,
    "data_parallel_size": DP,
    "max_num_seqs": MAX_NUM_SEQS,
    "max_model_len": MAX_MODEL_LEN,
    "quantization": "awq_marlin",
}


def cuda_driver_linker_environment() -> dict[str, str]:
    # Kaggle exposes libcuda.so.1 at runtime but its CUDA image can omit the
    # unversioned libcuda.so linker name needed by FlashInfer JIT.
    candidates = []
    ldconfig = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, check=False)
    for line in ldconfig.stdout.splitlines():
        if "libcuda.so.1" in line and "=>" in line:
            candidates.append(Path(line.rsplit("=>", 1)[1].strip()))
    candidates.extend(
        [
            Path("/usr/lib/x86_64-linux-gnu/libcuda.so.1"),
            Path("/usr/local/nvidia/lib64/libcuda.so.1"),
            Path("/usr/lib64/libcuda.so.1"),
        ]
    )
    driver_library = next((path for path in candidates if path.is_file()), None)
    if driver_library is None:
        raise RuntimeError("CUDA driver is active but libcuda.so.1 was not found.")

    linker_dir = Path("/kaggle/working/cuda-driver-link")
    linker_dir.mkdir(parents=True, exist_ok=True)
    linker_name = linker_dir / "libcuda.so"
    if linker_name.is_symlink() or linker_name.exists():
        linker_name.unlink()
    linker_name.symlink_to(driver_library.resolve())

    environment = os.environ.copy()
    for variable in ("LIBRARY_PATH", "LD_LIBRARY_PATH"):
        existing = [item for item in environment.get(variable, "").split(os.pathsep) if item]
        environment[variable] = os.pathsep.join(
            [str(linker_dir), *[item for item in existing if item != str(linker_dir)]]
        )

    probe_path = linker_dir / "cuda_link_probe"
    probe = subprocess.run(
        ["c++", "-x", "c++", "-", f"-L{linker_dir}", "-lcuda", "-o", str(probe_path)],
        input="int main() { return 0; }\n",
        capture_output=True,
        text=True,
        env=environment,
    )
    probe_path.unlink(missing_ok=True)
    if probe.returncode:
        raise RuntimeError(f"CUDA driver linker probe failed:\n{probe.stderr}")
    print("verified CUDA driver linker:", linker_name, "->", driver_library)
    return environment


def served_model_ids() -> set[str]:
    try:
        health = requests.get(f"{VLLM_BASE}/health", timeout=2)
        if not health.ok:
            return set()
        response = requests.get(f"{VLLM_BASE}/v1/models", timeout=5)
        response.raise_for_status()
        return {item["id"] for item in response.json().get("data", [])}
    except (requests.RequestException, KeyError, TypeError, ValueError):
        return set()


existing_models = served_model_ids()
if existing_models:
    assert (
        MODEL in existing_models
    ), f"Port 8000 already serves {sorted(existing_models)}, not {MODEL}. Restart the session."
    assert (
        VLLM_CONFIG.exists()
    ), "A pre-existing vLLM server has unknown TP/DP; restart the session."
    actual_server_config = json.loads(VLLM_CONFIG.read_text(encoding="utf-8"))
    assert (
        actual_server_config == expected_server_config
    ), f"Existing vLLM config {actual_server_config} != {expected_server_config}; restart."
    print("reusing healthy vLLM server:", sorted(existing_models))
else:
    server_environment = cuda_driver_linker_environment()
    server_log = open("/kaggle/working/vllm.log", "a", encoding="utf-8")  # noqa: SIM115
    serve_cmd = [
        "vllm",
        "serve",
        MODEL,
        "--served-model-name",
        MODEL,
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
        "--tensor-parallel-size",
        str(TP),
        "--data-parallel-size",
        str(DP),
        "--api-server-count",
        "1",
        "--dtype",
        "half",
        "--quantization",
        # vLLM prints "Detected that the model can run with awq_marlin, however you
        # specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin
        # for faster inference" -- the fast kernel was available on this T4 all along and the
        # explicit flag was turning it off. Decode ran at 3.33 tok/s, which is what made a
        # 2048-token budget unreachable inside a 360 s request timeout.
        "awq_marlin",
        "--max-model-len",
        "16384",
        "--max-num-seqs",
        str(MAX_NUM_SEQS),
        "--gpu-memory-utilization",
        "0.90",
        "--generation-config",
        "vllm",
        "--default-chat-template-kwargs",
        '{"enable_thinking": false}',
        "--seed",
        "20260802",
    ]
    if MODEL_REVISION:
        serve_cmd += ["--revision", MODEL_REVISION]
    server = subprocess.Popen(
        serve_cmd, stdout=server_log, stderr=subprocess.STDOUT, env=server_environment
    )

    for _ in range(120):
        if MODEL in served_model_ids():
            break
        if server.poll() is not None:
            log_tail = Path("/kaggle/working/vllm.log").read_text(
                encoding="utf-8", errors="replace"
            )[-50_000:]
            raise RuntimeError(log_tail)
        time.sleep(5)
    else:
        raise TimeoutError("vLLM did not become healthy; inspect /kaggle/working/vllm.log")
    VLLM_CONFIG.write_text(json.dumps(expected_server_config, indent=2) + "\n", encoding="utf-8")
    print("vLLM ready:", MODEL, MODEL_REVISION, "tp/dp=", TP, DP)

In [ ]:
# D1/D2: diagnose the unexplained generation slowdown before trusting a full run.
# Alternate constrained/unconstrained requests over the exact production prompt and stream
# them so TTFT is not mixed with decode speed. The output is diagnostic only.
PERF_DIAGNOSTIC = Path(
    f"/kaggle/working/artifacts/structured_decoding_benchmark_{MODEL_RUN_TAG}_{PROJECT_SHA[:12]}.json"
)
RUN_DIAGNOSTICS = os.environ.get("VIFINQA_RUN_DIAGNOSTICS") == "1"
if not RUN_DIAGNOSTICS:
    print("VIFINQA_RUN_DIAGNOSTICS=0: skipping the decoding benchmark (settled 14/08).")
else:
    perf_cmd = [
        sys.executable,
        "scripts/72_benchmark_structured_decoding.py",
        "--retrieval",
        str(RETRIEVAL),
        "--manifest",
        str(MANIFEST.with_suffix(".parquet")),
        "--data-root",
        str(DATA_ROOT),
        "--output",
        str(PERF_DIAGNOSTIC),
        "--model",
        MODEL,
        "--model-revision",
        MODEL_REVISION or "unknown",
        "--project-revision",
        PROJECT_SHA,
        "--table-unit-source",
        TABLE_UNIT_SOURCE,
        "--candidate-tables",
        os.environ.get("VIFINQA_CANDIDATE_TABLES", "20"),
        "--max-tokens",
        "128",
        "--repeats",
        "2",
        "--concurrency",
        str(min(2, MAX_NUM_SEQS)),
    ]
    for question_id in [1, 213, 399, 442, 473]:
        perf_cmd += ["--id", str(question_id)]
    subprocess.run(perf_cmd, check=True)

    # The quantisation and attention implementations are facts in the server log, not settings
    # we should infer from the GPU name. Preserve and print the discriminating lines.
    vllm_log = Path("/kaggle/working/vllm.log").read_text(encoding="utf-8", errors="replace")
    kernel_markers = ("quant", "awq", "marlin", "kernel", "attention backend", "turing")
    kernel_lines = [
        line for line in vllm_log.splitlines()
        if any(marker in line.casefold() for marker in kernel_markers)
    ]
    KERNEL_DIAGNOSTIC = Path(
        f"/kaggle/working/artifacts/vllm_kernel_diagnostic_{MODEL_RUN_TAG}_{PROJECT_SHA[:12]}.log"
    )
    KERNEL_DIAGNOSTIC.write_text("\n".join(kernel_lines) + "\n", encoding="utf-8")
    print("\n".join(kernel_lines[-100:]))
    assert kernel_lines, "vLLM log exposed no quantisation/kernel selection; inspect it manually."

In [ ]:
# Smoke five fixed cases: simple, USD, multi-company, percentage, and trillion-VND.
SMOKE_GEN = Path(
    f"/kaggle/working/artifacts/generation_{MODEL_RUN_TAG}_smoke_{PROJECT_SHA[:12]}"
)
SMOKE_IDS = [1, 213, 399, 442, 473]
smoke_cmd = [
    sys.executable,
    "scripts/50_generate_programs.py",
    "--retrieval",
    str(RETRIEVAL),
    "--manifest",
    str(MANIFEST.with_suffix(".parquet")),
    "--data-root",
    str(DATA_ROOT),
    "--output",
    str(SMOKE_GEN),
    "--model",
    MODEL,
    "--thinking-mode",
    THINKING_MODE,
    "--table-unit-source",
    TABLE_UNIT_SOURCE,
    "--max-attempts",
    "3",
    # Recall at depth 20 was 0.6283 while depth 100 reached 0.8612, so ten candidates
    # hid more than a third of the gold tables from the model (submissions 2805, 2810).
    "--candidate-tables",
    os.environ.get("VIFINQA_CANDIDATE_TABLES", "20"),
    "--max-tokens",
    MAX_TOKENS,
    "--context-limit",
    str(MAX_MODEL_LEN),
    "--project-revision",
    PROJECT_SHA,
]
for question_id in SMOKE_IDS:
    smoke_cmd += ["--id", str(question_id)]
if MODEL_REVISION:
    smoke_cmd += ["--model-revision", MODEL_REVISION]
if FINAL_RUN:
    smoke_cmd += ["--final-run"]
smoke_started = time.monotonic()
subprocess.run(smoke_cmd, check=True)
smoke_elapsed = time.monotonic() - smoke_started
smoke_errors = []
if (SMOKE_GEN / "errors.jsonl").exists():
    smoke_errors = [
        json.loads(line) for line in (SMOKE_GEN / "errors.jsonl").read_text().splitlines() if line
    ]
smoke_predictions = json.loads((SMOKE_GEN / "submission.json").read_text(encoding="utf-8"))
smoke_traces = []
if (SMOKE_GEN / "program_traces.jsonl").exists():
    smoke_traces = [
        json.loads(line)
        for line in (SMOKE_GEN / "program_traces.jsonl").read_text(encoding="utf-8").splitlines()
        if line
    ]
print(
    "smoke predictions/errors/traces:", len(smoke_predictions), len(smoke_errors), len(smoke_traces)
)
print(json.dumps(smoke_predictions[:2], ensure_ascii=False, indent=2)[:4000])
if smoke_errors:
    print("SMOKE ERRORS (full unresolved records):")
    print(json.dumps(smoke_errors, ensure_ascii=False, indent=2)[:30_000])
assert {row["id"] for row in smoke_predictions} == set(SMOKE_IDS)
assert {row["id"] for row in smoke_traces} == set(SMOKE_IDS)
smoke_fallbacks = [trace for trace in smoke_traces if trace.get("fallback")]
smoke_rescued = [trace for trace in smoke_traces if trace.get("rescued")]
print(f"fallback {len(smoke_fallbacks)}, rescued {len(smoke_rescued)} of {len(smoke_traces)}")
for trace in smoke_fallbacks:
    print(f"  fallback q{trace['id']}: {trace.get('fallback_reason')}")
# Counting was the wrong gate. These five IDs were picked as the hardest cases in the release,
# so a count says more about the sample than about the run. What matters is *which* question
# fails: 399, 442 and 473 are cohort programs over 8 to 32 ticker-year routes, and the 8B has
# failed them on every run so far. A fourth ID appearing here is a regression; those three
# appearing is Tuesday.
KNOWN_HARD = {399, 442, 473}
unexpected = sorted({int(trace["id"]) for trace in smoke_fallbacks} - KNOWN_HARD)
assert not unexpected, (
    f"Questions {unexpected} fell back and are not in the known-hard set; that is a regression "
    "in the infrastructure. Do not continue."
)
# The real regression test is below: two exact answers that only come out right when retrieval,
# grounding, units and execution all agree.
smoke_by_id = {int(row["id"]): row for row in smoke_predictions}
expected_smoke_answers = {1: 208253.201298, 213: 6.15569834}
for question_id, expected_answer in expected_smoke_answers.items():
    actual_answer = float(smoke_by_id[question_id]["answer"])
    assert math.isclose(actual_answer, expected_answer, rel_tol=1e-9, abs_tol=1e-6), (
        f"Smoke q{question_id}={actual_answer}, expected {expected_answer}. Stop."
    )
effective_parallelism = min(SHARDS, DP * MAX_NUM_SEQS)
projected_unit_branch_hours = (
    smoke_elapsed / len(SMOKE_IDS) * 200 / effective_parallelism / 3600
)
print(
    f"smoke wall={smoke_elapsed:.1f}s; conservative 200-question branch projection="
    f"{projected_unit_branch_hours:.2f}h at effective parallelism {effective_parallelism}"
)
SMOKE_GATE = SMOKE_GEN / "smoke_gate.json"
SMOKE_GATE.write_text(
    json.dumps(
        {
            "passed": True,
            "project_revision": PROJECT_SHA,
            "model": MODEL,
            "model_revision": MODEL_REVISION,
            "retrieval_sha256": sha256(RETRIEVAL),
            "question_ids": SMOKE_IDS,
            "expected_answers": expected_smoke_answers,
            "wall_seconds": smoke_elapsed,
            "effective_parallelism": effective_parallelism,
            "projected_unit_branch_hours": projected_unit_branch_hours,
        },
        ensure_ascii=False,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)
print("smoke gate passed:", SMOKE_GATE)

In [ ]:
# Widest-route gate: prove the hardest questions generate, and measure the pace they set.
WIDE_GEN = Path(
    f"/kaggle/working/artifacts/generation_{MODEL_RUN_TAG}_wide_{PROJECT_SHA[:12]}"
)
retrieval_rows = [
    json.loads(line) for line in RETRIEVAL.read_text(encoding="utf-8").splitlines() if line.strip()
]


def route_fan_out(row: dict[str, object]) -> int:
    # Routing expands to one candidate per ticker-year pair, and a cohort program needs one
    # operand per route, so this is the width the decoding grammar has to admit.
    spec = row["query_spec"]
    assert isinstance(spec, dict)
    return max(1, len(spec["tickers"])) * max(1, len(spec["years"]))


RUN_DIAGNOSTICS = os.environ.get("VIFINQA_RUN_DIAGNOSTICS") == "1"
if not RUN_DIAGNOSTICS:
    print("VIFINQA_RUN_DIAGNOSTICS=0: skipping the widest-route gate (cost 2.5h on 15/08).")
else:
    wide_ids = sorted(int(row["id"]) for row in retrieval_rows if route_fan_out(row) > 10)
    assert len(wide_ids) >= 20, f"Unexpected fan-out distribution: {len(wide_ids)} questions"
    print("widest questions:", len(wide_ids), "| max fan-out:", max(map(route_fan_out, retrieval_rows)))
    wide_cmd = [
        sys.executable,
        "scripts/50_generate_programs.py",
        "--retrieval",
        str(RETRIEVAL),
        "--manifest",
        str(MANIFEST.with_suffix(".parquet")),
        "--data-root",
        str(DATA_ROOT),
        "--output",
        str(WIDE_GEN),
        "--model",
        MODEL,
        "--thinking-mode",
        THINKING_MODE,
        "--table-unit-source",
        TABLE_UNIT_SOURCE,
        "--max-attempts",
        "3",
        # Recall at depth 20 was 0.6283 while depth 100 reached 0.8612, so ten candidates
        # hid more than a third of the gold tables from the model (submissions 2805, 2810).
        "--candidate-tables",
        os.environ.get("VIFINQA_CANDIDATE_TABLES", "20"),
        "--max-tokens",
        MAX_TOKENS,
        "--context-limit",
        str(MAX_MODEL_LEN),
        "--project-revision",
        PROJECT_SHA,
    ]
    for question_id in wide_ids:
        wide_cmd += ["--id", str(question_id)]
    if MODEL_REVISION:
        wide_cmd += ["--model-revision", MODEL_REVISION]
    wide_started = time.monotonic()
    subprocess.run(wide_cmd, check=True)
    wide_elapsed = time.monotonic() - wide_started
    wide_errors = []
    if (WIDE_GEN / "errors.jsonl").exists():
        wide_errors = [
            json.loads(line)
            for line in (WIDE_GEN / "errors.jsonl").read_text(encoding="utf-8").splitlines()
            if line
        ]
    wide_traces = [
        json.loads(line)
        for line in (WIDE_GEN / "program_traces.jsonl").read_text(encoding="utf-8").splitlines()
        if line
    ]
    if wide_errors:
        print("WIDE GATE ERRORS (full unresolved records):")
        print(json.dumps(wide_errors, ensure_ascii=False, indent=2)[:30_000])
    wide_rate = len(wide_ids) / max(wide_elapsed, 1e-9)
    print(
        f"wide gate: {len(wide_traces)}/{len(wide_ids)} traces, {len(wide_errors)} errors, "
        f"{wide_rate:.3f} questions/s, full run <= {1012 / (DP * wide_rate) / 3_600:.2f} h on {DP} "
        "shards. These are the heaviest prompts in the release, so that projection is a ceiling."
    )
    assert not wide_errors, "The widest questions must generate before a full run is worth starting."
    assert {int(row["id"]) for row in wide_traces} == set(wide_ids)

In [ ]:
# Representative sample: the smoke set is deliberately the hard tail, so it cannot say what
# the run will score or how long it will take. A seeded random draw, sharded exactly like the
# full run, can say both.
import random

SAMPLE_SIZE = int(os.environ.get("VIFINQA_SAMPLE_SIZE", "40"))
SAMPLE_GEN = Path(
    f"/kaggle/working/artifacts/generation_{MODEL_RUN_TAG}_sample_{PROJECT_SHA[:12]}"
)
retrieval_rows = [
    json.loads(line) for line in RETRIEVAL.read_text(encoding="utf-8").splitlines() if line.strip()
]
RUN_DIAGNOSTICS = os.environ.get("VIFINQA_RUN_DIAGNOSTICS") == "1"
if not RUN_DIAGNOSTICS:
    print("VIFINQA_RUN_DIAGNOSTICS=0: skipping the representative sample projection.")
else:
    sample_ids = sorted(
        random.Random(20260802).sample([int(row["id"]) for row in retrieval_rows], SAMPLE_SIZE)
    )
    sample_common = [
        sys.executable,
        "scripts/50_generate_programs.py",
        "--retrieval",
        str(RETRIEVAL),
        "--manifest",
        str(MANIFEST.with_suffix(".parquet")),
        "--data-root",
        str(DATA_ROOT),
        "--model",
        MODEL,
        "--thinking-mode",
        THINKING_MODE,
        "--table-unit-source",
        TABLE_UNIT_SOURCE,
        "--max-attempts",
        "3",
        # Recall at depth 20 was 0.6283 while depth 100 reached 0.8612, so ten candidates
        # hid more than a third of the gold tables from the model (submissions 2805, 2810).
        "--candidate-tables",
        os.environ.get("VIFINQA_CANDIDATE_TABLES", "20"),
        "--max-tokens",
        MAX_TOKENS,
        "--context-limit",
        str(MAX_MODEL_LEN),
        "--project-revision",
        PROJECT_SHA,
    ]
    for question_id in sample_ids:
        sample_common += ["--id", str(question_id)]
    if MODEL_REVISION:
        sample_common += ["--model-revision", MODEL_REVISION]

    sample_started = time.monotonic()
    sample_workers = []
    sample_dirs = [SAMPLE_GEN / f"shard_{index}" for index in range(SHARDS)]
    for shard_index, shard_dir in enumerate(sample_dirs):
        sample_workers.append(
            subprocess.Popen(
                sample_common
                + ["--output", str(shard_dir), "--shard-count", str(SHARDS), "--shard-index", str(shard_index)]
            )
        )
    for worker in sample_workers:
        assert worker.wait() == 0, "A sample shard failed; read its traceback above."
    sample_elapsed = time.monotonic() - sample_started


    def _read_jsonl(path: Path) -> list[dict[str, object]]:
        if not path.exists():
            return []
        return [
            json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()
        ]


    sample_traces = [row for path in sample_dirs for row in _read_jsonl(path / "program_traces.jsonl")]
    sample_errors = [row for path in sample_dirs for row in _read_jsonl(path / "errors.jsonl")]
    answered = len(sample_traces)
    print(f"answered {answered}/{len(sample_ids)} = {100 * answered / len(sample_ids):.0f}%")

    failures: dict[str, int] = {}
    for record in sample_errors:
        for attempt in record.get("failed_attempts", []):
            message = str(attempt.get("error", ""))[:70]
            failures[message] = failures.get(message, 0) + 1
    for reason, count in sorted(failures.items(), key=lambda item: -item[1])[:12]:
        print(f"  {count:3d}  {reason}")

    latencies = sorted(
        float(trace["latency_seconds"]) for trace in sample_traces if trace.get("latency_seconds")
    )
    tokens = [int(trace["completion_tokens"]) for trace in sample_traces if trace.get("completion_tokens")]
    if latencies:
        print(f"answered latency: median {latencies[len(latencies) // 2]:.1f}s, max {latencies[-1]:.1f}s")
    if tokens and latencies:
        print(f"emitted tokens: median {sorted(tokens)[len(tokens) // 2]}, max {max(tokens)}")
        print(f"decode rate per request: about {sum(tokens) / sum(latencies):.1f} tokens/s")
    per_question = sample_elapsed / len(sample_ids)
    print(
        f"wall clock {sample_elapsed / 60:.1f} min for {len(sample_ids)} questions on {SHARDS} shards "
        f"= {per_question:.1f}s each; full run about {1012 * per_question / 3600:.1f} h"
    )

In [ ]:
# Controlled 200-question ablation: frozen manifest defaults versus parser unit v3.
# This is opt-in because the two matched runs consume 400 generations.
RUN_SUBSET_200 = os.environ.get("VIFINQA_RUN_SUBSET_200") == "1"
# Opt-in, and a full run is meant to walk straight past it. Raising here stopped a 12-hour
# session between the smoke and the run it was preparing for, which is the opposite of what a
# guard on an optional experiment should do. It refuses only what it is actually asked to do.
if not RUN_SUBSET_200:
    print("RUN_MODE is not 'subset': skipping the 200-question unit ablation. Run the full cell.")
else:
    subset_expected_sha = os.environ.get("VIFINQA_EXPECTED_PROJECT_SHA", "").strip()
    assert subset_expected_sha and subset_expected_sha == PROJECT_SHA, (
        "Subset runs require VIFINQA_EXPECTED_PROJECT_SHA to match the published checkout."
    )
    assert PERF_DIAGNOSTIC.is_file() and KERNEL_DIAGNOSTIC.is_file(), (
        "Run D1/D2 and retain both diagnostics before the subset."
    )
    assert SMOKE_GATE.is_file(), "Run and pass the smoke gate before the subset."
    smoke_gate = json.loads(SMOKE_GATE.read_text(encoding="utf-8"))
    projected_branch_hours = float(smoke_gate["projected_unit_branch_hours"])
    if projected_branch_hours > 8:
        assert os.environ.get("VIFINQA_ALLOW_LONG_SUBSET") == "1", (
            f"Smoke projects {projected_branch_hours:.2f}h for one branch. Do not risk a "
            "cancelled Kaggle version; inspect D1/D2 or deliberately override the gate."
        )
    requested_unit_variants = tuple(
        item.strip()
        for item in os.environ.get("VIFINQA_UNIT_VARIANTS", "manifest").split(",")
        if item.strip()
    )
    assert requested_unit_variants and len(requested_unit_variants) == len(
        set(requested_unit_variants)
    )
    assert set(requested_unit_variants) <= {"manifest", "latest"}, (
        "VIFINQA_UNIT_VARIANTS must be manifest, latest, or manifest,latest."
    )
    SUBSET_CONFIG = Path("configs/experiment_subset_200.json")
    subset_config = json.loads(SUBSET_CONFIG.read_text(encoding="utf-8"))
    subset_ids = [int(question_id) for question_id in subset_config["ids"]]
    assert subset_config["size"] == 200 and len(subset_ids) == len(set(subset_ids)) == 200
    assert len(subset_config["source_sha256"]) == 64
    SUBSET_DIR_NAME = f"unit_ablation_subset_200_{MODEL_RUN_TAG}_{PROJECT_SHA[:12]}"
    SUBSET_ROOT = Path("/kaggle/working/artifacts") / SUBSET_DIR_NAME


    def make_subset_writable(root: Path) -> None:
        for path in [root, *root.rglob("*")]:
            path.chmod(path.stat().st_mode | (0o700 if path.is_dir() else 0o600))


    # A saved first branch can be attached to the second session. Import only a checkpoint
    # whose model and project fingerprint match this exact experiment.
    for unit_source in ("manifest", "latest"):
        variant_root = SUBSET_ROOT / unit_source
        prior_variant = [
            path
            for path in iter_input_paths(
                f"{SUBSET_DIR_NAME}/{unit_source}/shard_0/run_metadata.json"
            )
            if json.loads(path.read_text(encoding="utf-8")).get("project_revision")
            == PROJECT_SHA
            and json.loads(path.read_text(encoding="utf-8")).get("model") == MODEL
        ]
        assert len(prior_variant) <= 1, f"Ambiguous {unit_source} checkpoints: {prior_variant}"
        if not variant_root.exists() and prior_variant:
            shutil.copytree(prior_variant[0].parents[1], variant_root)
            make_subset_writable(variant_root)
            print("imported unit checkpoint:", unit_source, prior_variant[0].parents[1])


    def summarize_unit_variant(unit_source: str) -> dict[str, object]:
        merged = SUBSET_ROOT / unit_source / "merged"
        predictions = json.loads((merged / "submission.json").read_text(encoding="utf-8"))
        assert {int(row["id"]) for row in predictions} == set(subset_ids)
        return {
            "unit_source": unit_source,
            "questions": len(predictions),
            "submission_sha256": sha256(merged / "submission.json"),
            "run_metadata_sha256": sha256(merged / "run_metadata.json"),
            "output": str(merged),
        }


    def run_unit_variant(unit_source: str) -> None:
        variant_root = SUBSET_ROOT / unit_source
        shard_dirs = [variant_root / f"shard_{index}" for index in range(SHARDS)]
        common = [
            sys.executable,
            "scripts/50_generate_programs.py",
            "--retrieval",
            str(RETRIEVAL),
            "--manifest",
            str(MANIFEST.with_suffix(".parquet")),
            "--data-root",
            str(DATA_ROOT),
            "--model",
            MODEL,
            "--thinking-mode",
            THINKING_MODE,
            "--max-attempts",
            "3",
            "--candidate-tables",
            os.environ.get("VIFINQA_CANDIDATE_TABLES", "20"),
            "--max-tokens",
            MAX_TOKENS,
            "--context-limit",
            str(MAX_MODEL_LEN),
            "--project-revision",
            PROJECT_SHA,
            "--table-unit-source",
            unit_source,
        ]
        for question_id in subset_ids:
            common += ["--id", str(question_id)]
        if MODEL_REVISION:
            common += ["--model-revision", MODEL_REVISION]

        workers = []
        for shard_index, shard_dir in enumerate(shard_dirs):
            command = common + [
                "--output",
                str(shard_dir),
                "--shard-count",
                str(SHARDS),
                "--shard-index",
                str(shard_index),
            ]
            workers.append((shard_index, subprocess.Popen(command)))
        for shard_index, worker in workers:
            return_code = worker.wait()
            if return_code:
                raise subprocess.CalledProcessError(return_code, [unit_source, str(shard_index)])

        merged = variant_root / "merged"
        subprocess.run(
            [
                sys.executable,
                "scripts/51_merge_generation_shards.py",
                *[str(path) for path in shard_dirs],
                "--output",
                str(merged),
                "--expected-rows",
                "200",
            ],
            check=True,
        )
        print("completed unit variant:", summarize_unit_variant(unit_source))


    for mode in requested_unit_variants:
        run_unit_variant(mode)
    unit_ablation = {
        mode: summarize_unit_variant(mode)
        for mode in ("manifest", "latest")
        if (SUBSET_ROOT / mode / "merged/submission.json").is_file()
    }
    UNIT_ABLATION_MANIFEST = SUBSET_ROOT / "comparison_manifest.json"
    UNIT_ABLATION_MANIFEST.write_text(
        json.dumps(
            {
                "subset_config_sha256": sha256(SUBSET_CONFIG),
                "subset_source_sha256": subset_config["source_sha256"],
                "retrieval_sha256": sha256(RETRIEVAL),
                "project_revision": PROJECT_SHA,
                "model_profile": MODEL_PROFILE,
                "model": MODEL,
                "model_revision": MODEL_REVISION,
                "structured_decoding_benchmark_sha256": sha256(PERF_DIAGNOSTIC),
                "kernel_diagnostic_sha256": sha256(KERNEL_DIAGNOSTIC),
                "smoke_gate_sha256": sha256(SMOKE_GATE),
                "variants": unit_ablation,
            },
            ensure_ascii=False,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )
    print(UNIT_ABLATION_MANIFEST.read_text(encoding="utf-8"))
    OFFLINE_UNIT_COMPARISON = SUBSET_ROOT / "offline_comparison.json"
    if set(unit_ablation) == {"manifest", "latest"}:
        subprocess.run(
            [
                sys.executable,
                "scripts/54_compare_generation_variants.py",
                str(SUBSET_ROOT / "manifest/merged"),
                str(SUBSET_ROOT / "latest/merged"),
                "--output",
                str(OFFLINE_UNIT_COMPARISON),
            ],
            check=True,
        )
        print(OFFLINE_UNIT_COMPARISON.read_text(encoding="utf-8"))
    else:
        missing_variants = sorted({"manifest", "latest"} - set(unit_ablation))
        print(
            "Save this Kaggle version, attach its output to the next session, then run:",
            "VIFINQA_UNIT_VARIANTS=" + ",".join(missing_variants),
        )

In [ ]:
# Full resume-safe final generation. Enable both flags in config, rerun config, then this cell.
RUN_FULL = os.environ.get("VIFINQA_RUN_FULL") == "1"
FINAL_RUN = os.environ.get("VIFINQA_FINAL_RUN") == "1"
assert RUN_FULL, "Set VIFINQA_RUN_FULL=1 only after inspecting smoke output."
assert FINAL_RUN, "Set VIFINQA_FINAL_RUN=1 for the pinned submission-candidate run."
# What a final run must guarantee is that its code came from a published commit, and cells 2 and
# 3 already establish that: they fetch VIFINQA_GIT_REF from origin and record the SHA they
# checked out. Demanding a hand-pasted copy of that same SHA added no guarantee and one failure
# mode -- the value can never be this notebook's own commit, because committing the notebook
# moves it -- and it stopped a session between the smoke and the run it was preparing for.
#
# So the declaration is optional and enforced when made, exactly as in cells 2 and 3. What is not
# optional is having a real checkout to record.
final_expected_sha = os.environ.get("VIFINQA_EXPECTED_PROJECT_SHA", "").strip()
if final_expected_sha:
    assert final_expected_sha == PROJECT_SHA, (
        f"Kaggle checked out {PROJECT_SHA}, expected {final_expected_sha}. "
        "Stop: this is not the published experiment snapshot."
    )
else:
    print(f"final run pinned to {PROJECT_SHA} from ref {GIT_REF}; record it with the results.")
if MODEL_TOTAL_PARAMS_B > 14:
    assert os.environ.get("VIFINQA_ORGANIZER_CONFIRMED_14B") == "1", (
        "Final 14B-profile run requires written organiser confirmation."
    )
assert THINKING_MODE == "disabled", "Final run requires Qwen3 non-thinking mode."
assert len(PROJECT_SHA) == 40, "Final run requires a recorded project Git SHA."
dense_config = json.loads((DENSE / "config.json").read_text(encoding="utf-8"))
assert (
    dense_config.get("model_revision") == DENSE_REVISION
), "Dense index was not built with the approved BGE-M3 revision."
# Reranking is optional, so pin its provenance only when the run actually used it. A final
# run on the hybrid ranking is a legitimate choice; a final run on an unrecorded one is not.
retrieval_metadata_path = RETRIEVAL.with_suffix(RETRIEVAL.suffix + ".metadata.json")
assert retrieval_metadata_path.is_file(), "The retrieval used by a final run must be recorded."
retrieval_metadata = json.loads(retrieval_metadata_path.read_text(encoding="utf-8"))
if RETRIEVAL != HYBRID:
    assert retrieval_metadata.get("model_revision") == RERANKER_REVISION, retrieval_metadata
    assert retrieval_metadata.get("project_revision") == RERANK_PROJECT_SHA, retrieval_metadata
    assert retrieval_metadata.get("max_length") == 8192, retrieval_metadata
    assert retrieval_metadata.get("candidate_tables") == 100, retrieval_metadata
print("final run retrieval:", RETRIEVAL.name)
GENERATION_NAME = f"generation_{MODEL_RUN_TAG}"
GEN = Path("/kaggle/working/artifacts") / GENERATION_NAME
GEN_SHARDS = Path("/kaggle/working/artifacts") / f"{GENERATION_NAME}_shards"
prior_generation = [
    path
    for path in iter_input_paths(f"{GENERATION_NAME}_shards/shard_0/run_metadata.json")
    if json.loads(path.read_text(encoding="utf-8")).get("project_revision") == PROJECT_SHA
    and json.loads(path.read_text(encoding="utf-8")).get("model") == MODEL
]
if not GEN_SHARDS.exists() and prior_generation:
    prior_shards = prior_generation[0].parents[1]
    shutil.copytree(prior_shards, GEN_SHARDS)
    print("imported prior generation checkpoints:", prior_shards)


def completed_rows() -> int:
    """Count answers the way the merge counts them.

    Globbing the checkpoint directory reported zero on a run whose shards had all just
    reported completion, which turned a finished run into a failed session. The consolidated
    files are what the merge reads, so let them be what decides whether a run is done.
    """
    total = 0
    for shard in sorted(GEN_SHARDS.glob("shard_*")):
        seen: set[int] = set()
        for name in ("predictions.jsonl", "errors.jsonl"):
            path = shard / name
            if not path.is_file():
                continue
            for line in path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    seen.add(int(json.loads(line)["id"]))
        if not seen:
            seen = {int(item.stem) for item in (shard / "completed/rows").glob("*.json")}
        total += len(seen)
    return total


# A Kaggle session is shorter than this run. Rows are written one at a time and
# atomically, so a session that dies loses at most the questions in flight, and the
# next one resumes from whatever survived.
print(f"resuming with {completed_rows()}/1012 questions already answered")
common_cmd = [
    sys.executable,
    "scripts/50_generate_programs.py",
    "--retrieval",
    str(RETRIEVAL),
    "--manifest",
    str(MANIFEST.with_suffix(".parquet")),
    "--data-root",
    str(DATA_ROOT),
    "--model",
    MODEL,
    "--thinking-mode",
    THINKING_MODE,
    "--table-unit-source",
    TABLE_UNIT_SOURCE,
    "--max-attempts",
    "3",
    # Recall at depth 20 was 0.6283 while depth 100 reached 0.8612, so ten candidates
    # hid more than a third of the gold tables from the model (submissions 2805, 2810).
    "--candidate-tables",
    os.environ.get("VIFINQA_CANDIDATE_TABLES", "20"),
    "--max-tokens",
    MAX_TOKENS,
    "--context-limit",
    str(MAX_MODEL_LEN),
    "--project-revision",
    PROJECT_SHA,
]
if MODEL_REVISION:
    common_cmd += ["--model-revision", MODEL_REVISION]
common_cmd += ["--final-run"]
QUESTION_LIMIT = os.environ.get("VIFINQA_QUESTION_LIMIT", "").strip()
if QUESTION_LIMIT:
    common_cmd += ["--limit", QUESTION_LIMIT]
shard_dirs = [GEN_SHARDS / f"shard_{index}" for index in range(SHARDS)]
workers = []
for shard_index, shard_dir in enumerate(shard_dirs):
    shard_cmd = common_cmd + [
        "--output",
        str(shard_dir),
        "--shard-count",
        str(SHARDS),
        "--shard-index",
        str(shard_index),
    ]
    workers.append((shard_index, subprocess.Popen(shard_cmd)))
for shard_index, worker in workers:
    return_code = worker.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, ["generation-shard", str(shard_index)])
answered = completed_rows()
if QUESTION_LIMIT:
    print(f"partial run finished cleanly: {answered} answered under a {QUESTION_LIMIT}-question cap.")
    print("Save this version. Its output is what the finishing notebook resumes from.")
elif answered < 1012:
    raise SystemExit(
        f"Session ended with {answered}/1012 answered. Nothing is lost: save this\n"
        "notebook version, start a new session, attach this output as an input, and\n"
        "run the same cells. The checkpoint is keyed to the project revision, so do\n"
        "not change code or shard count between sessions."
    )
else:
    subprocess.run(
        [
            sys.executable,
            "scripts/51_merge_generation_shards.py",
            *[str(path) for path in shard_dirs],
            "--output",
            str(GEN),
            "--expected-rows",
            "1012",
        ],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "scripts/45_finalize_submission.py",
            str(GEN / "submission.json"),
            "--retrieval",
            str(RETRIEVAL),
            "--manifest",
            str(MANIFEST.with_suffix(".parquet")),
            "--output",
            str(GEN / "submission_z4_abs.json"),
        ],
        check=True,
    )
    FINAL_SUBMISSION = GEN / "submission_z4_abs.json"
    subprocess.run(
        [
            sys.executable,
            "scripts/40_validate_submission.py",
            str(FINAL_SUBMISSION),
            "--questions",
            str(DATA_ROOT / "questions/questions.jsonl"),
            "--evidence-root",
            str(GEN),
            "--allow-partial-docs",
        ],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "scripts/41_package_submission.py",
            str(FINAL_SUBMISSION),
            "/kaggle/working/submission.zip",
            "--questions",
            str(DATA_ROOT / "questions/questions.jsonl"),
            "--evidence-root",
            str(GEN),
            "--allow-partial-docs",
        ],
        check=True,
    )
    submission_zip = Path("/kaggle/working/submission.zip")
    final_predictions = json.loads(FINAL_SUBMISSION.read_text(encoding="utf-8"))
    final_errors = []
    if (GEN / "errors.jsonl").exists():
        final_errors = [
            line for line in (GEN / "errors.jsonl").read_text(encoding="utf-8").splitlines() if line
        ]
    # Completeness is the gate, not perfection: the organiser discards a file with any
    # question missing, while a question the model never solved still carries its retrieval
    # citation. Fallbacks are recorded in the traces, so count them rather than forbid them.
    final_traces = [
        json.loads(line)
        for line in (GEN / "program_traces.jsonl").read_text(encoding="utf-8").splitlines()
        if line
    ]
    fallbacks = [trace for trace in final_traces if trace.get("fallback")]
    print(
        f"predictions {len(final_predictions)}/1012, solved "
        f"{len(final_predictions) - len(fallbacks)}, fallback {len(fallbacks)}, "
        f"unresolved errors {len(final_errors)}"
    )
    assert len(final_predictions) == 1012
    FINAL_MANIFEST = Path("/kaggle/working/final_artifacts.json")
    final_manifest = {
        "project_revision": PROJECT_SHA,
        "model_profile": MODEL_PROFILE,
        "model": MODEL,
        "model_revision": MODEL_REVISION,
        "model_total_parameters_billions": MODEL_TOTAL_PARAMS_B,
        "model_non_embedding_parameters_billions": MODEL_NON_EMBEDDING_PARAMS_B,
        "organizer_confirmed_14b": os.environ.get("VIFINQA_ORGANIZER_CONFIRMED_14B") == "1",
        "dense_revision": DENSE_REVISION,
        "reranker": RERANKER,
        "reranker_revision": RERANKER_REVISION,
        "hybrid_project_revision": HYBRID_PROJECT_SHA,
        "reranker_project_revision": RERANK_PROJECT_SHA,
        "thinking_mode": THINKING_MODE,
        "table_unit_source": TABLE_UNIT_SOURCE,
        "tensor_parallel_size": TP,
        "data_parallel_size": DP,
        "hybrid_retrieval_sha256": sha256(HYBRID),
        "retrieval_sha256": sha256(RETRIEVAL),
        "submission_json_sha256": sha256(FINAL_SUBMISSION),
        "submission_profile": "z4_abs_line",
        "submission_zip_sha256": sha256(submission_zip),
        "runtime_environment_sha256": sha256(RUNTIME_LOG),
        "structured_decoding_benchmark_sha256": (sha256(PERF_DIAGNOSTIC) if PERF_DIAGNOSTIC.exists() else None),
        "kernel_diagnostic_sha256": (sha256(KERNEL_DIAGNOSTIC) if KERNEL_DIAGNOSTIC.exists() else None),
        "smoke_gate_sha256": sha256(SMOKE_GATE),
        "questions": len(final_predictions),
    }
    FINAL_MANIFEST.write_text(
        json.dumps(final_manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print(submission_zip, submission_zip.stat().st_size, final_manifest["submission_zip_sha256"])
    print(FINAL_MANIFEST, json.dumps(final_manifest, indent=2))
